In [1]:
"""
The purpose of this Jupyter notebook is to refine the raw screen
intensities via multiplication by the VACV host-factor probability.
"""

'\nThe purpose of this Jupyter notebook is to refine the raw screen\nintensities via multiplication by the VACV host-factor probability.\n'

In [2]:
import os

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Loading Data

In [ ]:
# Load the probabilities into a DataFrame
probs_entire_screen_path = (
    "data_for_refinement_of_entire_screen/all_hf_entire_screen_"
    "predictions_PPI_only_model.tsv"
)

probs_entire_screen_df = pd.read_csv(
    probs_entire_screen_path,
    sep="\t"
)

# Load the TSV file with the mean intensities
mean_ints_path = (
    "data_for_refinement_of_entire_screen/screen_subset_mean_features.tsv"
)

mean_ints_df = pd.read_csv(
    mean_ints_path,
    sep="\t"
)

In [4]:
# Extract the intensities
# Bear in mind that the Voronoi cell intensities are used
# Also remember to remove duplicate entries
early_ints_df = mean_ints_df[[
    "Name",
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"
]].drop_duplicates("Name")

late_ints_df = mean_ints_df[[
    "Name",
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"
]].drop_duplicates("Name")

# Extract the probabilities
probs_df = probs_entire_screen_df[["gene", "probability"]]

# Intensity Refinement via Probability Multiplication

In [5]:
# Multiply the mean intensities by the probabilities predicted by the PU
# learning model
# To this end, a Series is created mapping the gene names to the
# predicted probabilities
prob_map = probs_df.set_index("gene")["probability"]

refined_early_ints_df = early_ints_df[["Name"]].copy()
refined_early_ints_df["refined_intensity"] = (
    early_ints_df["dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"]
    *
    early_ints_df["Name"].map(prob_map)
)

refined_late_ints_df = late_ints_df[["Name"]].copy()
refined_late_ints_df["refined_intensity"] = (
    late_ints_df["dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"]
    *
    late_ints_df["Name"].map(prob_map)
)

In [7]:
# Perform a quick sanity check: Verify that after multiplication, the
# maximum value has decreased and the minimum value has increased
early_max_val_unrefined = early_ints_df[
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"
].max()
early_min_val_unrefined = early_ints_df[
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"
].min()

late_max_val_unrefined = late_ints_df[
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"
].max()
late_min_val_unrefined = late_ints_df[
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"
].min()


early_max_val_refined = refined_early_ints_df["refined_intensity"].max()
early_min_val_refined = refined_early_ints_df["refined_intensity"].min()

late_max_val_refined = refined_late_ints_df["refined_intensity"].max()
late_min_val_refined = refined_late_ints_df["refined_intensity"].min()

assert (
    (early_max_val_unrefined > early_max_val_refined)
    and
    (early_min_val_unrefined < early_min_val_refined)
    and
    (late_max_val_unrefined > late_max_val_refined)
    and
    (late_min_val_unrefined < late_min_val_refined)
), (
    "Something went wring while performing the refinement!"
)

# Saving the Intensities to Disk

In [8]:
# Save the unrefined as well as the refined intensities to disk
# To this end, comparison DataFrames are created, i.e. DataFrames
# containing both the unrefined and the refined intensities
comparison_early_ints = early_ints_df[["Name"]].copy()
comparison_early_ints["unrefined_intensity"] = early_ints_df[
    "dIntensity_cPathogen_eMean_oVoronoiCells_nZScore"
]
comparison_early_ints["refined_intensity"] = refined_early_ints_df[
    "refined_intensity"
]

comparison_late_ints = late_ints_df[["Name"]].copy()
comparison_late_ints["unrefined_intensity"] = late_ints_df[
    "dIntensity_cLatePathogen_eMean_oVoronoiCells_nZScore"
]
comparison_late_ints["refined_intensity"] = refined_late_ints_df[
    "refined_intensity"
]

In [ ]:
# Save the comparison DataFrames to disk
comparison_early_ints.to_csv(
    "unrefined_and_refined_early_intensities.tsv",
    sep="\t",
    index=False
)

comparison_late_ints.to_csv(
    "unrefined_and_refined_late_intensities.tsv",
    sep="\t",
    index=False
)